In [107]:

import pandas as pd
import torch
from pandas import DataFrame
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM

# Data preparation

In [108]:
# Model parameters
HORIZON = 1
BATCH_SIZE = 800
NUM_EPOCHS = 25
HIDDEN_SIZE = 128
N_LAYERS = 3
DROPOUT = 0.3
EMBEDDING_SIZE = 32

# Train parameters
TARGET = "EXPORT_centered"
FEATURES = [
  "contig", "comlang_off", "colony", "smctry",  # dist cepii categorical
]
N_SPLITS = 5
PATIENCE = 5
LEARNING_RATE = 0.01
WEIGHT_DECAY = 0.01
RANDOM_SEED = 16
KEEP_FRAC = 1.0
N_LAGS = 5
SUBSAMPLE_ENABLED = False
N_DYADS = 1000

SANCTION_COLS = ["arms", "military", "trade", "travel", "other"]

# Torch config
torch.manual_seed(RANDOM_SEED)
device = (
  torch.device("mps") if torch.backends.mps.is_available()
  else torch.device("cpu")
)

In [109]:
processed = pd.read_parquet(path="../../data/model/processed.parquet", engine="fastparquet")

df: DataFrame = processed.copy(deep=True)
df["dyad_id"] = df["ISO3_reporter"] + "_" + df["ISO3_partner"]
df = df.sort_values(by=["dyad_id", "Year"], ignore_index=True)

# Prepare sanction column as sum of all active boolean sanctions
df["sanction"] = (df[SANCTION_COLS]
                  .sum(axis=1)).astype(int)

# Coerce numerical columns to float
num_cols = ["distw", "GDP_reporter", "GDP_partner", "sanction", "contig",
            "comlang_off", "colony", "smctry", "Year", ]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(float)

# Drop NA in numerical columns
df = df.dropna(subset=num_cols)

# Cast "Year" to integer
df["Year"] = df["Year"].astype(int)

# Cast "dyad_id" to categorical
for col in ["dyad_id"]:
  df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()))

  # Define columns to be lagged and lag them while appending the number of the lag to the column name
lag_cols = ["GDP_reporter", "GDP_partner", "sanction"]
for col in lag_cols:
  for index in range(1, N_LAGS + 1):
    df[f"{col}_lag{index}"] = df.groupby("dyad_id", observed=True)[col].shift(index)

# Drop NA again for the lags that produced NA
df = df.dropna()

# Add lagged column names to the feature list
FEATURES += [f"{c}_lag{index}" for c in lag_cols for index in range(1, N_LAGS + 1)]

In [110]:
DYAD_PAIRS = [
  ("USA_CHN", "CHN_USA"),
  ("USA_CAN", "CAN_USA"),
  ("DEU_CHN", "CHN_DEU"),
  ("USA_DEU", "DEU_USA"),
  ("DEU_FRA", "FRA_DEU"),
]

# Check time series for stationarity and cointegration

In [142]:
# Define which dyad to investigate
dyad_id = "DEU_CHN"
dyad_df = df[df["dyad_id"] == dyad_id].sort_values("Year").copy()
dyad_df = dyad_df.set_index(pd.PeriodIndex(dyad_df["Year"], freq="Y"))

# Define which time series to investigate
ts_columns = [
  "GDP_reporter",
  "GDP_partner",
  "sanction",
  "EXPORT"
]

In [143]:
# Check for stationarity using Augmented Dickey-Fuller test
for col in ts_columns:

  try:
    result_adf = adfuller(dyad_df[col].values, autolag="AIC")
  except ValueError as e:
    print("!" * 50)
    print(f"⚠️ Column \"{col}\" constant over the whole dyad! Skipping!!")
    print(f"!" * 50 + "\n\n")
    continue

  adf_statistic = result_adf[0]
  p_value = result_adf[1]
  critical_values = result_adf[4]

  print(f"ADF Test for time series: {col}")
  print("=" * 50)

  print(f"p-value: {p_value}")
  print(f"ADF statistic: {adf_statistic}")
  print(
    f"Critical value 1%: {critical_values["1%"]}\nCritical value 5%: {critical_values["5%"]}\nCritical value 10%: {critical_values["10%"]}\n\n")

ADF Test for time series: GDP_reporter
p-value: 0.46677853089768784
ADF statistic: -1.6313607343858119
Critical value 1%: -3.7883858816542486
Critical value 5%: -3.013097747543462
Critical value 10%: -2.6463967573696143


ADF Test for time series: GDP_partner
p-value: 1.0
ADF statistic: 3.5172624062716515
Critical value 1%: -3.8092091249999998
Critical value 5%: -3.0216450000000004
Critical value 10%: -2.6507125


ADF Test for time series: sanction
p-value: 0.971636826596387
ADF statistic: 0.18898223650461382
Critical value 1%: -3.6889256286443146
Critical value 5%: -2.9719894897959187
Critical value 10%: -2.6252957653061224


ADF Test for time series: EXPORT
p-value: 0.9514605909693693
ADF statistic: -0.07931155367495603
Critical value 1%: -3.8092091249999998
Critical value 5%: -3.0216450000000004
Critical value 10%: -2.6507125




In [144]:
# Check for stationarity using Kwiatkowski-Phillips-Schmidt-Shin test
for col in ts_columns:

  try:
    result_adf = kpss(dyad_df[col].values, regression="ct")
  except ValueError as error:
    print("!" * 50)
    print(f"⚠️ Column \"{col}\" constant over the whole dyad! Skipping!!")
    print(f"!" * 50 + "\n\n")
    continue

  kpss_statistic = result_adf[0]
  p_value = result_adf[1]
  critical_values = result_adf[3]

  print(f"KPSS Test for time series: {col}")
  print("=" * 50)

  print(f"p-value: {p_value}")
  print(f"KPSS statistic: {kpss_statistic}")
  print(
    f"Critical value 1%: {critical_values["1%"]}\nCritical value 2.5%: {critical_values["2.5%"]}\nCritical value 5%: {critical_values["5%"]}\nCritical value 10%: {critical_values["10%"]}\n\n")

KPSS Test for time series: GDP_reporter
p-value: 0.1
KPSS statistic: 0.08402665821387242
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


KPSS Test for time series: GDP_partner
p-value: 0.01
KPSS statistic: 0.21936577229818938
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


KPSS Test for time series: sanction
p-value: 0.049209478117167284
KPSS statistic: 0.14694862625939925
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119


KPSS Test for time series: EXPORT
p-value: 0.08852631962441393
KPSS statistic: 0.12519578740281648
Critical value 1%: 0.216
Critical value 2.5%: 0.176
Critical value 5%: 0.146
Critical value 10%: 0.119




/var/folders/wz/kf7643gn3_s2867t_nnc9zxc0000gn/T/ipykernel_8611/2801000533.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result_adf = kpss(dyad_df[col].values, regression="ct")
/var/folders/wz/kf7643gn3_s2867t_nnc9zxc0000gn/T/ipykernel_8611/2801000533.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result_adf = kpss(dyad_df[col].values, regression="ct")


In [145]:
columns_coint = ["EXPORT", "GDP_reporter_lag2"]
columns_exog = ["sanction_lag1"]

In [146]:
# Using VAR to calculate the optimal amount of lags for< Johansen Test and VECM
var_data = dyad_df[columns_coint].dropna()
var_model = VAR(var_data)
var_selection = var_model.select_order(5)
var_result = var_model.fit(var_selection.aic)
k_ar_diff = max(var_result.k_ar - 1, 1)
k_ar_diff

2

In [147]:
# Check for cointegration using Johansen test
data_johansen = dyad_df[columns_coint].dropna()

jres = coint_johansen(data_johansen, det_order=1, k_ar_diff=k_ar_diff)
trace_statistic = jres.lr1[0]
max_eigenvalue_statistic = jres.lr2[0]
print(f"Johansen Test for cointegration between {columns_coint[0]} and {columns_coint[1]} reporter")
print("=" * 50)
print("Trace Statistics:", jres.lr1)
print("Critical Values (Trace):", jres.cvt)

trace_stats = jres.lr1
crit_vals = jres.cvt

# Pick confidence level column (0=90%, 1=95%, 2=99%)
alpha_col = 1  # for 99% significance
rank = sum(trace_stats > crit_vals[:, alpha_col])
print(f"Estimated cointegration rank at 99%: {rank}")

Johansen Test for cointegration between EXPORT and GDP_reporter_lag2 reporter
Trace Statistics: [17.10392688  5.88824752]
Critical Values (Trace): [[16.1619 18.3985 23.1485]
 [ 2.7055  3.8415  6.6349]]
Estimated cointegration rank at 99%: 1


# VECM (Vector Error Correction Models)

Because GDP and EXPORT are both non-stationary and cointegrated, we cannot run the normal Granger causality test. But, we can run VECM.

In [148]:
Y = dyad_df[columns_coint].dropna()
X_exog = dyad_df[columns_exog].reindex(Y.index).fillna(0).astype(float)
vecm = VECM(endog=Y, exog=X_exog, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic="ci")
vecm_result = vecm.fit()
vecm_result.summary()

,coef,std err,z,P>|z|,[0.025,0.975]
exog1,-4.125e+06,2.51e+06,-1.641,0.101,-9.05e+06,8.02e+05
L1.EXPORT,0.1937,0.201,0.962,0.336,-0.201,0.588
L1.GDP_reporter_lag2,6.043e-06,6.53e-06,0.925,0.355,-6.76e-06,1.88e-05
L2.EXPORT,-0.1010,0.183,-0.553,0.580,-0.459,0.257
L2.GDP_reporter_lag2,-6.504e-06,6.59e-06,-0.986,0.324,-1.94e-05,6.42e-06
,coef,std err,z,P>|z|,[0.025,0.975]
exog1,1.564e+11,5.72e+10,2.734,0.006,4.43e+10,2.69e+11
L1.EXPORT,-6912.0361,4584.338,-1.508,0.132,-1.59e+04,2073.101
L1.GDP_reporter_lag2,0.3200,0.149,2.152,0.031,0.029,0.611
L2.EXPORT,1.993e+04,4158.557,4.793,0.000,1.18e+04,2.81e+04
